# Projet Final : Biais et Équité dans la Classification d'Images Médicales
## NIH Chest X-ray — ResNet18

### Trinôme : Othmane Nammous, Tharushan Uthayakumar, Rémi Malapert

---

## 1. Introduction

Ce projet s'inscrit dans l'étude de l'équité (fairness) en intelligence artificielle appliquée à l'imagerie médicale. Nous utilisons un sous-ensemble du dataset NIH Chest X-ray 14, composé d'images de radiographies thoraciques compressées en $256 \times 256$ pixels.

Objectif : Évaluer et corriger les biais d'un modèle de classification d'images (ResNet18) qui prédit si un patient est malade ou sain à partir de la seule image radiographique.

Particularité du modèle : Le ResNet18 ne reçoit que l'image en entrée, sans métadonnée explicite. Il peut néanmoins déduire des attributs sensibles comme le genre, la position PA/AP et l'âge. Si ces attributs sont déséquilibrés dans les données, le modèle risque de reproduire ces biais.

Approche :
1. Analyse des données : identifier les déséquilibres factuels.
2. Pre-processing : tester plusieurs pondérations (genre, position, genre × position).
3. Post-processing : appliquer ROC et Calibrated Equalized Odds.
4. Combinaisons : comparer pre seul, post seul, et pre+post.
5. Analyse : discuter le compromis performance/fairness à partir des résultats exécutés.

In [37]:
# === Imports ===
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os
from datetime import datetime

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", append=True, category=UserWarning)

# Métriques de fairness (AIF360 sklearn API)
from aif360.sklearn.metrics import (
    statistical_parity_difference,
    disparate_impact_ratio,
    equal_opportunity_difference,
    average_odds_difference,
    base_rate,
    smoothed_edf,
    df_bias_amplification,
    conditional_demographic_disparity,
)
from sklearn.metrics import balanced_accuracy_score, accuracy_score

# AIF360 pour le post-processing
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric
from aif360.algorithms.postprocessing.reject_option_classification import RejectOptionClassification
from aif360.algorithms.postprocessing.calibrated_eq_odds_postprocessing import CalibratedEqOddsPostprocessing

# Entraînement du modèle
from train_classifieur import train_classifier, pred_classifier
import torch

In [38]:
import os
print(os.getcwd())

d:\CPES\L3\Projet Fairness


In [39]:
# === Configuration ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device utilisé : {device}")
if device.type == 'cuda':
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"Mémoire allouée : {round(torch.cuda.memory_allocated(0)/1024**3,1)} GB")

# Chemins
BASE_DIR = r"D:\CPES\L3\Projet Fairness"
DATA_DIR = BASE_DIR + r"\Malapert_Remi"
EXPE_DIR = os.path.join(BASE_DIR, "expe_log")
CSV_PATH = DATA_DIR + r"\metadata.csv"
os.makedirs(EXPE_DIR, exist_ok=True)

# Paramètres d'entraînement
# Rendu final: garder False.
# Debug local rapide: mettre True pour 1 époque.
SMOKE_TEST = True
MAX_EPOCHS = 1 if SMOKE_TEST else 25
print(f"Mode smoke test: {SMOKE_TEST} | max_epochs={MAX_EPOCHS}")

Device utilisé : cpu
Mode smoke test: True | max_epochs=1


---
## 2. Préparation et Analyse des Données

Nous disposons d'un dataset personnel de radiographies thoraciques, accompagné d'un fichier `metadata.csv` contenant les métadonnées de chaque image (genre, âge, position, pathologie, etc.).

L'analyse ici est **plus concise** que celle du mi-projet : nous mettons en évidence les **déséquilibres majeurs** qui justifieront les corrections appliquées par la suite.

In [40]:
# === Chargement des données ===
df = pd.read_csv(CSV_PATH)
print(f"Nombre total d'images : {len(df)}")
print(f"Colonnes disponibles : {list(df.columns)}")
print()

# === Statistiques descriptives ===
print("=" * 50)
print("RÉPARTITION TRAIN / VALIDATION")
print(df['train_valid'].value_counts().to_string())
print()

print("RÉPARTITION DES LABELS")
print(df['label'].value_counts().to_string())
print(f"  → Taux de malades : {(df['label'] == 'malade').mean():.2%}")
print()

print("RÉPARTITION PAR GENRE")
print(df['Patient Gender'].value_counts().to_string())
print(f"  → Proportion d'hommes : {(df['Patient Gender'] == 'M').mean():.2%}")
print()

print("RÉPARTITION PAR POSITION")
print(df['View Position'].value_counts().to_string())
print(f"  → Proportion de PA : {(df['View Position'] == 'PA').mean():.2%}")
print()

print("STATISTIQUES SUR L'ÂGE")
print(df['Patient Age'].describe().to_string())

Nombre total d'images : 5222
Colonnes disponibles : ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11', 'train_valid', 'label', 'WEIGHTS']

RÉPARTITION TRAIN / VALIDATION
train_valid
train    3976
valid    1246

RÉPARTITION DES LABELS
label
sain      2858
malade    2364
  → Taux de malades : 45.27%

RÉPARTITION PAR GENRE
Patient Gender
M    3110
F    2112
  → Proportion d'hommes : 59.56%

RÉPARTITION PAR POSITION
View Position
PA    3200
AP    2022
  → Proportion de PA : 61.28%

STATISTIQUES SUR L'ÂGE
count    5222.000000
mean       44.992149
std        18.289353
min         2.000000
25%        30.000000
50%        46.000000
75%        58.000000
max       412.000000


In [41]:
# === Visualisation des déséquilibres clés ===

# Encodage binaire pour l'analyse
df['label_binary'] = (df['label'] == 'malade').astype(int)
df['Gender_Binary'] = (df['Patient Gender'] == 'M').astype(int)   # M=1 (privilégié), F=0
df['Position_Binary'] = (df['View Position'] == 'PA').astype(int)  # PA=1 (privilégié), AP=0

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Label par Genre (%)", "Label par Position (%)",
        "Distribution de l'âge par Genre", "Taux de maladie par Genre et Position"
    ],
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "histogram"}, {"type": "bar"}]]
)

# 1) Label × Genre
ct_gender = pd.crosstab(df['Patient Gender'], df['label'], normalize='index') * 100
fig.add_trace(go.Bar(name='malade', x=ct_gender.index.tolist(), y=ct_gender['malade'].tolist(),
                     marker_color='#EF553B', showlegend=True), row=1, col=1)
fig.add_trace(go.Bar(name='sain', x=ct_gender.index.tolist(), y=ct_gender['sain'].tolist(),
                     marker_color='#636EFA', showlegend=True), row=1, col=1)

# 2) Label × Position
ct_pos = pd.crosstab(df['View Position'], df['label'], normalize='index') * 100
fig.add_trace(go.Bar(name='malade', x=ct_pos.index.tolist(), y=ct_pos['malade'].tolist(),
                     marker_color='#EF553B', showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(name='sain', x=ct_pos.index.tolist(), y=ct_pos['sain'].tolist(),
                     marker_color='#636EFA', showlegend=False), row=1, col=2)

# 3) Distribution d'âge par genre
for gender, color in [('M', '#636EFA'), ('F', '#EF553B')]:
    ages = df[df['Patient Gender'] == gender]['Patient Age']
    fig.add_trace(go.Histogram(x=ages, name=gender, opacity=0.6, nbinsx=30,
                               marker_color=color), row=2, col=1)

# 4) Taux de maladie par genre × position
ct_cross = df.groupby(['Patient Gender', 'View Position'])['label_binary'].mean().reset_index()
ct_cross['group'] = ct_cross['Patient Gender'] + ' / ' + ct_cross['View Position']
fig.add_trace(go.Bar(x=ct_cross['group'].tolist(), y=ct_cross['label_binary'].tolist(),
                     marker_color=['#636EFA', '#AB63FA', '#EF553B', '#FFA15A'],
                     showlegend=False), row=2, col=2)

fig.update_layout(barmode='stack', height=700, title_text="Déséquilibres dans le dataset",
                  template='plotly_white')
fig.show()

In [42]:
# === Métriques de fairness sur les données brutes (labels réels) ===

print("=" * 60)
print("FAIRNESS DES LABELS — Genre (M=privilégié vs F=défavorisé)")
print("=" * 60)

spd_g = statistical_parity_difference(
    y_true=df['label_binary'], prot_attr=df['Gender_Binary'], pos_label=1)
dir_g = disparate_impact_ratio(
    y_true=df['label_binary'], prot_attr=df['Gender_Binary'], pos_label=1)
print(f"  Statistical Parity Difference (SPD) : {spd_g:.4f}")
print(f"  Disparate Impact Ratio (DIR)        : {dir_g:.4f}")

print()
print("=" * 60)
print("FAIRNESS DES LABELS — Position (PA=privilégié vs AP=défavorisé)")
print("=" * 60)

spd_p = statistical_parity_difference(
    y_true=df['label_binary'], prot_attr=df['Position_Binary'], pos_label=1)
dir_p = disparate_impact_ratio(
    y_true=df['label_binary'], prot_attr=df['Position_Binary'], pos_label=1)
print(f"  Statistical Parity Difference (SPD) : {spd_p:.4f}")
print(f"  Disparate Impact Ratio (DIR)        : {dir_p:.4f}")

print()
print("Rappel : Un DIR entre 0.8 et 1.25 indique une parité approximative (règle des 80%).")
print(f"  → Genre   : {'✓ Parité OK' if 0.8 <= dir_g <= 1.25 else '✗ BIAIS DÉTECTÉ'}")
print(f"  → Position : {'✓ Parité OK' if 0.8 <= dir_p <= 1.25 else '✗ BIAIS DÉTECTÉ'}")

FAIRNESS DES LABELS — Genre (M=privilégié vs F=défavorisé)
  Statistical Parity Difference (SPD) : -0.0359
  Disparate Impact Ratio (DIR)        : 0.9232

FAIRNESS DES LABELS — Position (PA=privilégié vs AP=défavorisé)
  Statistical Parity Difference (SPD) : 0.0909
  Disparate Impact Ratio (DIR)        : 1.2177

Rappel : Un DIR entre 0.8 et 1.25 indique une parité approximative (règle des 80%).
  → Genre   : ✓ Parité OK
  → Position : ✓ Parité OK


---
## 3. Fonctions Utilitaires

Nous définissons ici les fonctions réutilisées tout au long du notebook, **adaptées des TDs de fairness** (TD3, TD4).

In [43]:
def get_metrics(y_true, y_pred=None, prot_attr=None, priv_group=1, pos_label=1, sample_weight=None):
    """
    Calcule un ensemble complet de métriques de fairness et de performance.
    Adapté du TD3/TD4 (aif360 sklearn API).

    Args:
        y_true:   array-like de vérité terrain (0/1)
        y_pred:   array-like de prédictions (0/1)
        prot_attr: array-like de l'attribut sensible
        priv_group: valeur du groupe privilégié
        pos_label: valeur du label positif
        sample_weight: poids des échantillons (optionnel)

    Returns:
        dict de métriques
    """
    group_metrics = {}
    group_metrics["base_rate_truth"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["statistical_parity_difference"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
        pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["disparate_impact_ratio"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
        pos_label=pos_label, sample_weight=sample_weight
    )
    if y_pred is not None:
        group_metrics["base_rate_preds"] = base_rate(
            y_true=y_pred, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
            pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
            pos_label=pos_label, sample_weight=sample_weight
        )
        if len(set(y_pred)) > 1:
            group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
                y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
                pos_label=pos_label, sample_weight=sample_weight
            )
        else:
            group_metrics["conditional_demographic_disparity"] = None
        group_metrics["smoothed_edf"] = smoothed_edf(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
            pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
            pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["balanced_accuracy_score"] = balanced_accuracy_score(
            y_true=y_true, y_pred=y_pred, sample_weight=sample_weight
        )
    return group_metrics

print("✓ Fonction get_metrics définie.")

✓ Fonction get_metrics définie.


In [44]:
def compute_reweighing_weights(df_in, label_col, protected_col):
    """
    Calcule les poids de reweighing pour rendre le label indépendant de l'attribut sensible.

    Formule : W(S=s, Y=y) = P(Y=y) × P(S=s) / P(Y=y, S=s)

    Args:
        df_in: DataFrame
        label_col: nom de la colonne label (binaire 0/1)
        protected_col: nom de la colonne de l'attribut sensible (binaire 0/1)

    Returns:
        Series de poids, indexée comme df_in
    """
    n = len(df_in)
    weights = pd.Series(1.0, index=df_in.index)

    for s_val in df_in[protected_col].unique():
        for y_val in df_in[label_col].unique():
            mask = (df_in[protected_col] == s_val) & (df_in[label_col] == y_val)
            n_sy = mask.sum()
            n_s = (df_in[protected_col] == s_val).sum()
            n_y = (df_in[label_col] == y_val).sum()

            if n_sy > 0:
                w = (n_y / n) * (n_s / n) / (n_sy / n)
                weights[mask] = w

    return weights


def apply_weights_to_csv(csv_in, csv_out, weights, weights_col='WEIGHTS'):
    """Sauvegarde le CSV avec les nouveaux poids dans la colonne WEIGHTS."""
    df_tmp = pd.read_csv(csv_in)
    df_tmp[weights_col] = weights.values
    df_tmp.to_csv(csv_out, index=False)
    print(f"  CSV sauvegardé : {csv_out}")
    print(f"  Poids — min={weights.min():.4f}, max={weights.max():.4f}, mean={weights.mean():.4f}")
    return csv_out

print("✓ Fonctions compute_reweighing_weights et apply_weights_to_csv définies.")

✓ Fonctions compute_reweighing_weights et apply_weights_to_csv définies.


In [45]:
def prepare_predictions(preds_csv_path):
    """
    Charge le CSV de prédictions et prépare les colonnes binaires pour l'analyse.

    Returns:
        DataFrame enrichi (label_binary, preds_binary, Gender_Binary, Position_Binary, score_malade)
    """
    df_p = pd.read_csv(preds_csv_path)

    # Encodage binaire
    df_p['label_binary'] = (df_p['labels'] == 'malade').astype(int)
    df_p['preds_binary'] = (df_p['preds'] == 'malade').astype(int)
    df_p['Gender_Binary'] = (df_p['Patient Gender'] == 'M').astype(int)
    df_p['Position_Binary'] = (df_p['View Position'] == 'PA').astype(int)

    # Conversion logits → probabilités via softmax
    logit_cols = [c for c in df_p.columns if c.startswith('preds_logit')]
    if len(logit_cols) >= 2:
        logits = torch.tensor(df_p[logit_cols].values.astype(float))
        probs = torch.softmax(logits, dim=1).numpy()
        df_p['score_malade'] = probs[:, 0]   # P(malade) — classe 0 dans ImageFolder
        df_p['score_sain'] = probs[:, 1]     # P(sain)

    return df_p


def evaluate_model(df_p, split='valid', prot_attr='Gender_Binary', priv_group=1, label=''):
    """
    Évalue les métriques de fairness et performance sur un split donné.

    Returns:
        dict de métriques
    """
    df_eval = df_p[df_p['train_valid'] == split].copy()

    metrics = get_metrics(
        y_true=df_eval['label_binary'].values,
        y_pred=df_eval['preds_binary'].values,
        prot_attr=df_eval[prot_attr].values,
        priv_group=priv_group,
        pos_label=1
    )
    metrics['method'] = label
    metrics['protected_attribute'] = prot_attr

    return metrics


def print_metrics(metrics, title=""):
    """Affiche proprement un dictionnaire de métriques."""
    if title:
        print(f"\n{'=' * 60}")
        print(title)
        print('=' * 60)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:45s} : {v:+.4f}" if 'ratio' not in k else f"  {k:45s} : {v:.4f}")


def create_aif360_dataset(df_eval, label_col='label_binary', prot_attr='Gender_Binary'):
    """Crée un BinaryLabelDataset AIF360 à partir d'un DataFrame."""
    df_aif = df_eval[[label_col, prot_attr]].copy().reset_index(drop=True)
    dataset = BinaryLabelDataset(
        df=df_aif,
        label_names=[label_col],
        protected_attribute_names=[prot_attr],
        favorable_label=1.0,
        unfavorable_label=0.0
    )
    return dataset

print("✓ Fonctions prepare_predictions, evaluate_model, print_metrics, create_aif360_dataset définies.")

✓ Fonctions prepare_predictions, evaluate_model, print_metrics, create_aif360_dataset définies.


---
## 4. Modèle de Base (Baseline)

Nous entraînons d'abord le modèle ResNet18 **sans aucune correction** (tous les poids `WEIGHTS = 1`), afin d'établir une **référence** (*baseline*) pour la performance et les biais.

> **Note :** L'entraînement prend environ 20-30 min sur CPU, ~10 min avec GPU.

In [46]:
%%time
# === Entraînement du modèle Baseline (WEIGHTS = 1 pour tout le monde) ===

dt = datetime.now()
logdir_baseline = f"{EXPE_DIR}/baseline_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"

ckpt_path_baseline, ckpt_score_baseline = train_classifier(
    logdir=logdir_baseline,
    datadir=DATA_DIR,
    csv=CSV_PATH,
    weights_col="WEIGHTS",
    max_epochs=MAX_EPOCHS,
)

print(f"\nModèle sauvegardé : {ckpt_path_baseline}")
print(f"Meilleur score (val_loss) : {ckpt_score_baseline:.4f}")

D:\CPES\L3\Projet Fairness\Malapert_Remi\metadata.csv D:\CPES\L3\Projet Fairness\expe_log/baseline_2026_03_18_16_21_00/csv_in_WEIGHTS.csv
num_workers set to : 0


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


End of training 356.41824555397034

Modèle sauvegardé : D:\CPES\L3\Projet Fairness\expe_log\baseline_2026_03_18_16_21_00\best-val-loss.ckpt
Meilleur score (val_loss) : 0.6281
CPU times: total: 42min 15s
Wall time: 5min 56s


In [47]:
%%time
# === Prédictions du modèle Baseline ===
BASELINE_PREDS = f"{logdir_baseline}/preds_baseline.csv"

pred_classifier(
    datadir=DATA_DIR,
    csv_in=CSV_PATH,
    csv_out=BASELINE_PREDS,
    ckpt_path=ckpt_path_baseline
)
print(f"\nPrédictions sauvegardées : {BASELINE_PREDS}")

Start prediction on train dataset
num_workers set to : 0


100%|██████████| 249/249 [02:06<00:00,  1.97it/s]


Predictions done in 126.61008596420288
Start prediction on validation dataset
num_workers set to : 0


100%|██████████| 78/78 [00:39<00:00,  1.99it/s]


Predictions done in 165.84496688842773
Global (train+validation) balanced accuracy without weigths 0.6531393162423523
Global (train+validation) accuracy without weigths 0.6643048640367675

Prédictions sauvegardées : D:\CPES\L3\Projet Fairness\expe_log/baseline_2026_03_18_16_21_00/preds_baseline.csv
CPU times: total: 21min 30s
Wall time: 4min 48s


In [48]:
# === Évaluation du modèle Baseline ===
# Sécurité: recharge les fonctions utilitaires si le kernel a été redémarré.
if 'prepare_predictions' not in globals() or 'evaluate_model' not in globals():
    raise RuntimeError(
        "Les fonctions utilitaires ne sont pas chargées. Exécuter d'abord les cellules 9, 10 et 11."
    )

df_baseline = prepare_predictions(BASELINE_PREDS)

# Balanced Accuracy globale (validation)
df_val_base = df_baseline[df_baseline['train_valid'] == 'valid']
ba_baseline = balanced_accuracy_score(df_val_base['label_binary'], df_val_base['preds_binary'])
print(f"Balanced Accuracy (validation) : {ba_baseline:.4f}")

# Métriques par Genre
metrics_baseline_gender = evaluate_model(df_baseline, prot_attr='Gender_Binary', label='Baseline')
print_metrics(metrics_baseline_gender, "Baseline — Fairness par Genre (M vs F)")

# Métriques par Position
metrics_baseline_pos = evaluate_model(df_baseline, prot_attr='Position_Binary', label='Baseline (pos)')
print_metrics(metrics_baseline_pos, "Baseline — Fairness par Position (PA vs AP)")

Balanced Accuracy (validation) : 0.6133

Baseline — Fairness par Genre (M vs F)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.0629
  disparate_impact_ratio                        : 0.8282
  base_rate_preds                               : +0.3371
  equal_opportunity_difference                  : -0.0727
  average_odds_difference                       : -0.0762
  conditional_demographic_disparity             : -0.0052
  smoothed_edf                                  : +0.1880
  df_bias_amplification                         : +0.0514
  balanced_accuracy_score                       : +0.6133

Baseline — Fairness par Position (PA vs AP)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.3413
  disparate_impact_ratio                        : 2.6466
  base_rate_preds                               : +0.3371
  equal_opportunity_difference                  : +0.3365
  avera

---
## 5. Pre-processing : Pondération (Reweighing)

Le pre-processing consiste à modifier les poids d'entraînement des échantillons pour compenser les déséquilibres. Le script train_classifieur.py utilise ces poids via un WeightedRandomSampler.

Nous testons trois stratégies de pondération :
1. Reweighing par Genre : corriger le biais lié au genre (M/F).
2. Reweighing par Position : corriger le biais lié à la position (PA/AP).
3. Reweighing croisé Genre × Position : corriger simultanément les deux axes.

### 5.1 Reweighing par Genre

In [49]:
# === Calcul des poids de Reweighing (Genre) ===
# Sécurité: recharge le DataFrame si le kernel a été redémarré.
if 'df' not in globals():
    df = pd.read_csv(CSV_PATH)
if 'label_binary' not in df.columns:
    df['label_binary'] = (df['label'] == 'malade').astype(int)
if 'Gender_Binary' not in df.columns:
    df['Gender_Binary'] = (df['Patient Gender'] == 'M').astype(int)

print("Poids de Reweighing par Genre :")
weights_gender = compute_reweighing_weights(df, 'label_binary', 'Gender_Binary')

for g in [0, 1]:
    for l in [0, 1]:
        mask = (df['Gender_Binary'] == g) & (df['label_binary'] == l)
        g_name = 'M' if g == 1 else 'F'
        l_name = 'malade' if l == 1 else 'sain'
        print(f"  Genre={g_name}, Label={l_name} : poids={weights_gender[mask].iloc[0]:.4f} (n={mask.sum()})")

# Sauvegarder le CSV
CSV_GENDER = f"{DATA_DIR}/metadata_rw_gender.csv"
apply_weights_to_csv(CSV_PATH, CSV_GENDER, weights_gender)

Poids de Reweighing par Genre :
  Genre=F, Label=sain : poids=0.9624 (n=1201)
  Genre=F, Label=malade : poids=1.0495 (n=911)
  Genre=M, Label=sain : poids=1.0272 (n=1657)
  Genre=M, Label=malade : poids=0.9690 (n=1453)
  CSV sauvegardé : D:\CPES\L3\Projet Fairness\Malapert_Remi/metadata_rw_gender.csv
  Poids — min=0.9624, max=1.0495, mean=1.0000


'D:\\CPES\\L3\\Projet Fairness\\Malapert_Remi/metadata_rw_gender.csv'

In [50]:
%%time
# === Entraînement avec poids Reweighing Genre ===
dt = datetime.now()
logdir_gender = f"{EXPE_DIR}/rw_gender_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"

ckpt_path_gender, ckpt_score_gender = train_classifier(
    logdir=logdir_gender,
    datadir=DATA_DIR,
    csv=CSV_GENDER,
    weights_col="WEIGHTS",
    max_epochs=MAX_EPOCHS,
)
print(f"\nModèle RW Genre sauvegardé : {ckpt_path_gender}")
print(f"Meilleur score (val_loss) : {ckpt_score_gender:.4f}")

GPU available: False, used: False


D:\CPES\L3\Projet Fairness\Malapert_Remi/metadata_rw_gender.csv D:\CPES\L3\Projet Fairness\expe_log/rw_gender_2026_03_18_16_32_47/csv_in_WEIGHTS.csv
num_workers set to : 0


TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


End of training 285.3483564853668

Modèle RW Genre sauvegardé : D:\CPES\L3\Projet Fairness\expe_log\rw_gender_2026_03_18_16_32_47\best-val-loss.ckpt
Meilleur score (val_loss) : 0.6546
CPU times: total: 36min 43s
Wall time: 4min 45s


In [51]:
%%time
# === Prédictions du modèle Reweighing Genre ===
GENDER_PREDS = f"{logdir_gender}/preds_rw_gender.csv"

pred_classifier(
    datadir=DATA_DIR,
    csv_in=CSV_PATH,  # On prédit sur les données originales
    csv_out=GENDER_PREDS,
    ckpt_path=ckpt_path_gender
)

Start prediction on train dataset
num_workers set to : 0


100%|██████████| 249/249 [01:45<00:00,  2.36it/s]


Predictions done in 105.56049919128418
Start prediction on validation dataset
num_workers set to : 0


100%|██████████| 78/78 [00:33<00:00,  2.36it/s]


Predictions done in 138.59960389137268
Global (train+validation) balanced accuracy without weigths 0.6601569021679283
Global (train+validation) accuracy without weigths 0.6464955955572578
CPU times: total: 18min 20s
Wall time: 2min 19s


In [52]:
# === Évaluation du modèle Reweighing Genre ===
df_gender = prepare_predictions(GENDER_PREDS)
metrics_rw_gender = evaluate_model(df_gender, prot_attr='Gender_Binary', label='RW Genre')
print_metrics(metrics_rw_gender, "Reweighing Genre — Fairness par Genre (M vs F)")


Reweighing Genre — Fairness par Genre (M vs F)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0552
  disparate_impact_ratio                        : 1.0926
  base_rate_preds                               : +0.6220
  equal_opportunity_difference                  : +0.0309
  average_odds_difference                       : +0.0384
  conditional_demographic_disparity             : +0.0043
  smoothed_edf                                  : +0.1468
  df_bias_amplification                         : +0.0102
  balanced_accuracy_score                       : +0.6353


### 5.2 Reweighing par Position (PA/AP)

La position de la radiographie (PA : Postéro-Antérieur, AP : Antéro-Postérieur) est fortement corrélée à l'état du patient dans notre dataset —  les patients AP sont souvent alités et plus gravement malades. On applique la même méthode de reweighing sur cet attribut.

In [53]:
# === Calcul des poids de Reweighing (Position) ===
# Sécurité: recharge le DataFrame si le kernel a été redémarré.
if 'df' not in globals():
    df = pd.read_csv(CSV_PATH)
if 'label_binary' not in df.columns:
    df['label_binary'] = (df['label'] == 'malade').astype(int)
if 'Position_Binary' not in df.columns:
    df['Position_Binary'] = (df['View Position'] == 'PA').astype(int)

print("Poids de Reweighing par Position :")
weights_position = compute_reweighing_weights(df, 'label_binary', 'Position_Binary')

for p in [0, 1]:
    for l in [0, 1]:
        mask = (df['Position_Binary'] == p) & (df['label_binary'] == l)
        p_name = 'PA' if p == 1 else 'AP'
        l_name = 'malade' if l == 1 else 'sain'
        print(f"  Position={p_name}, Label={l_name} : poids={weights_position[mask].iloc[0]:.4f} (n={mask.sum()})")

CSV_POSITION = f"{DATA_DIR}/metadata_rw_position.csv"
apply_weights_to_csv(CSV_PATH, CSV_POSITION, weights_position)

Poids de Reweighing par Position :
  Position=AP, Label=sain : poids=1.1133 (n=994)
  Position=AP, Label=malade : poids=0.8904 (n=1028)
  Position=PA, Label=sain : poids=0.9396 (n=1864)
  Position=PA, Label=malade : poids=1.0843 (n=1336)
  CSV sauvegardé : D:\CPES\L3\Projet Fairness\Malapert_Remi/metadata_rw_position.csv
  Poids — min=0.8904, max=1.1133, mean=1.0000


'D:\\CPES\\L3\\Projet Fairness\\Malapert_Remi/metadata_rw_position.csv'

In [54]:
%%time
# === Entraînement avec poids Reweighing Position ===
dt = datetime.now()
logdir_pos = f"{EXPE_DIR}/rw_position_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"

ckpt_path_pos, ckpt_score_pos = train_classifier(
    logdir=logdir_pos,
    datadir=DATA_DIR,
    csv=CSV_POSITION,
    weights_col="WEIGHTS"
)
print(f"\nModèle RW Position sauvegardé : {ckpt_path_pos}")
print(f"Meilleur score (val_loss) : {ckpt_score_pos:.4f}")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


D:\CPES\L3\Projet Fairness\Malapert_Remi/metadata_rw_position.csv D:\CPES\L3\Projet Fairness\expe_log/rw_position_2026_03_18_16_39_52/csv_in_WEIGHTS.csv
num_workers set to : 0
num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

End of training 11677.482643842697

Modèle RW Position sauvegardé : D:\CPES\L3\Projet Fairness\expe_log\rw_position_2026_03_18_16_39_52\best-val-loss.ckpt
Meilleur score (val_loss) : 0.5938
CPU times: total: 7h 6min 47s
Wall time: 3h 14min 37s


In [55]:
%%time
# === Prédictions du modèle Reweighing Position ===
POS_PREDS = f"{logdir_pos}/preds_rw_position.csv"

pred_classifier(
    datadir=DATA_DIR,
    csv_in=CSV_PATH,
    csv_out=POS_PREDS,
    ckpt_path=ckpt_path_pos
)

Start prediction on train dataset
num_workers set to : 0


100%|██████████| 249/249 [02:18<00:00,  1.80it/s]


Predictions done in 138.1597216129303
Start prediction on validation dataset
num_workers set to : 0


100%|██████████| 78/78 [00:39<00:00,  1.96it/s]


Predictions done in 178.03225326538086
Global (train+validation) balanced accuracy without weigths 0.6419055247892638
Global (train+validation) accuracy without weigths 0.6627728839525087
CPU times: total: 22min 51s
Wall time: 3min 2s


In [56]:
# === Évaluation du modèle Reweighing Position ===
df_pos = prepare_predictions(POS_PREDS)
metrics_rw_pos = evaluate_model(df_pos, prot_attr='Position_Binary', label='RW Position')
print_metrics(metrics_rw_pos, "Reweighing Position — Fairness par Position (PA vs AP)")

# Aussi évaluer par genre (pour voir l'impact croisé)
metrics_rw_pos_gender = evaluate_model(df_pos, prot_attr='Gender_Binary', label='RW Position (genre)')
print_metrics(metrics_rw_pos_gender, "Reweighing Position — Fairness par Genre")


Reweighing Position — Fairness par Position (PA vs AP)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.1021
  disparate_impact_ratio                        : 0.6317
  base_rate_preds                               : +0.2384
  equal_opportunity_difference                  : -0.1790
  average_odds_difference                       : -0.1160
  conditional_demographic_disparity             : -0.0317
  smoothed_edf                                  : +0.4565
  df_bias_amplification                         : +0.4021
  balanced_accuracy_score                       : +0.6267

Reweighing Position — Fairness par Genre
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0790
  disparate_impact_ratio                        : 1.3913
  base_rate_preds                               : +0.2384
  equal_opportunity_difference                  : +0.0414
  average_odds_difference         

### 5.3 Reweighing croisé Genre × Position

Pour tenir la promesse méthodologique de trois pondérations, on construit un attribut sensible croisé combinant genre et position.

- Groupes possibles : F_AP, F_PA, M_AP, M_PA
- Objectif : compenser les combinaisons sous-représentées du couple (attribut croisé, label)


In [57]:
%%time
# === Reweighing croisé: poids + entraînement + prédiction + évaluation ===
if 'df' not in globals():
    df = pd.read_csv(CSV_PATH)
if 'label_binary' not in df.columns:
    df['label_binary'] = (df['label'] == 'malade').astype(int)
if 'Gender_Binary' not in df.columns:
    df['Gender_Binary'] = (df['Patient Gender'] == 'M').astype(int)
if 'Position_Binary' not in df.columns:
    df['Position_Binary'] = (df['View Position'] == 'PA').astype(int)

# 0=F_AP, 1=F_PA, 2=M_AP, 3=M_PA
df['GenderPosition_Binary'] = 2 * df['Gender_Binary'] + df['Position_Binary']
weights_cross = compute_reweighing_weights(df, 'label_binary', 'GenderPosition_Binary')
CSV_CROSS = f"{DATA_DIR}/metadata_rw_gender_position.csv"
apply_weights_to_csv(CSV_PATH, CSV_CROSS, weights_cross)

dt = datetime.now()
logdir_cross = f"{EXPE_DIR}/rw_gender_position_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"
ckpt_path_cross, ckpt_score_cross = train_classifier(
    logdir=logdir_cross,
    datadir=DATA_DIR,
    csv=CSV_CROSS,
    weights_col="WEIGHTS",
    max_epochs=MAX_EPOCHS,
)
print(f"\nModèle RW croisé sauvegardé : {ckpt_path_cross}")
print(f"Meilleur score (val_loss) : {ckpt_score_cross:.4f}")

CROSS_PREDS = f"{logdir_cross}/preds_rw_gender_position.csv"
pred_classifier(
    datadir=DATA_DIR,
    csv_in=CSV_PATH,
    csv_out=CROSS_PREDS,
    ckpt_path=ckpt_path_cross
)

df_cross = prepare_predictions(CROSS_PREDS)
metrics_rw_cross_gender = evaluate_model(df_cross, prot_attr='Gender_Binary', label='RW Genre×Position')
metrics_rw_cross_pos = evaluate_model(df_cross, prot_attr='Position_Binary', label='RW Genre×Position')
print_metrics(metrics_rw_cross_gender, "RW Genre×Position — Fairness par Genre")
print_metrics(metrics_rw_cross_pos, "RW Genre×Position — Fairness par Position")

  CSV sauvegardé : D:\CPES\L3\Projet Fairness\Malapert_Remi/metadata_rw_gender_position.csv
  Poids — min=0.8552, max=1.1628, mean=1.0000
D:\CPES\L3\Projet Fairness\Malapert_Remi/metadata_rw_gender_position.csv D:\CPES\L3\Projet Fairness\expe_log/rw_gender_position_2026_03_18_19_57_32/csv_in_WEIGHTS.csv
num_workers set to : 0


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

End of training 286.55827045440674

Modèle RW croisé sauvegardé : D:\CPES\L3\Projet Fairness\expe_log\rw_gender_position_2026_03_18_19_57_32\best-val-loss.ckpt
Meilleur score (val_loss) : 0.6222
Start prediction on train dataset
num_workers set to : 0


100%|██████████| 249/249 [01:45<00:00,  2.36it/s]


Predictions done in 105.5346794128418
Start prediction on validation dataset
num_workers set to : 0


100%|██████████| 78/78 [00:33<00:00,  2.35it/s]


Predictions done in 138.74443888664246
Global (train+validation) balanced accuracy without weigths 0.6595737437821108
Global (train+validation) accuracy without weigths 0.6616238988893144

RW Genre×Position — Fairness par Genre
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0103
  disparate_impact_ratio                        : 1.0242
  base_rate_preds                               : +0.4278
  equal_opportunity_difference                  : -0.0196
  average_odds_difference                       : -0.0066
  conditional_demographic_disparity             : +0.0008
  smoothed_edf                                  : +0.0240
  df_bias_amplification                         : -0.1126
  balanced_accuracy_score                       : +0.6282

RW Genre×Position — Fairness par Position
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.1456
  disparate_impact_ratio          

### 5.4 Comparaison des méthodes de Pre-processing

Nous comparons les quatre approches (Baseline, RW Genre, RW Position, RW Genre × Position) selon la balanced accuracy et les principales métriques de fairness sur deux axes sensibles: Genre et Position.

In [58]:
# === Tableau comparatif Pre-processing ===
required = ['df_baseline', 'df_gender', 'df_pos', 'df_cross']
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"Variables manquantes pour la comparaison pre-processing: {missing}")

all_pre_metrics = []
for df_pred, name, prot in [
    (df_baseline, 'Baseline', 'Gender_Binary'),
    (df_baseline, 'Baseline', 'Position_Binary'),
    (df_gender, 'RW Genre', 'Gender_Binary'),
    (df_gender, 'RW Genre', 'Position_Binary'),
    (df_pos, 'RW Position', 'Gender_Binary'),
    (df_pos, 'RW Position', 'Position_Binary'),
    (df_cross, 'RW Genre×Position', 'Gender_Binary'),
    (df_cross, 'RW Genre×Position', 'Position_Binary'),
]:
    m = evaluate_model(df_pred, prot_attr=prot, label=name)
    m['attr'] = 'Genre' if 'Gender' in prot else 'Position'
    all_pre_metrics.append(m)

df_pre = pd.DataFrame(all_pre_metrics)
cols = [
    'method', 'attr', 'balanced_accuracy_score',
    'statistical_parity_difference', 'disparate_impact_ratio',
    'equal_opportunity_difference', 'average_odds_difference'
]

print("=== Tableau comparatif — Pre-processing ===")
display(df_pre[cols].sort_values(['attr', 'method']).round(4))

=== Tableau comparatif — Pre-processing ===


,method,attr,balanced_accuracy_score,statistical_parity_difference,disparate_impact_ratio,equal_opportunity_difference,average_odds_difference
0,Baseline,Genre,0.6133,-0.0629,0.8282,-0.0727,-0.0762
2,RW Genre,Genre,0.6353,0.0552,1.0926,0.0309,0.0384
6,RW Genre×Position,Genre,0.6282,0.0103,1.0242,-0.0196,-0.0066
4,RW Position,Genre,0.6267,0.0790,1.3913,0.0414,0.0616
1,Baseline,Position,0.6133,0.3413,2.6466,0.3365,0.3361
3,RW Genre,Position,0.6353,0.1402,1.2466,0.0979,0.1299
7,RW Genre×Position,Position,0.6282,-0.1456,0.6986,-0.1752,-0.1543
5,RW Position,Position,0.6267,-0.1021,0.6317,-0.1790,-0.1160


---
## 6. Post-processing

Le post-processing modifie les prédictions après entraînement pour améliorer l'équité sans modifier les poids du réseau.

Méthodes testées (TD4):
1. Reject Option Classification (ROC): ajuste seuil + marge sous contrainte de parité statistique.
2. Calibrated Equalized Odds: ajuste probabilistiquement les sorties pour réduire les écarts d'erreurs (ici contrainte FNR).

Couverture des expériences:
- Attributs sensibles testés: Genre et Position
- Modèles testés: Baseline, RW Genre, RW Position
- Objectif: comparer post seul vs combinaisons pre+post sur plusieurs critères.

In [59]:
def apply_post_processing(df_preds, method='ROC', prot_attr='Gender_Binary', label=''):
    """
    Applique une méthode de post-processing sur les prédictions (split validation uniquement).

    Args:
        df_preds: DataFrame avec prédictions et scores
        method: 'ROC' ou 'CalibratedEqOdds'
        prot_attr: attribut sensible binaire
        label: nom de la configuration

    Returns:
        dict de métriques après post-processing
    """
    df_val = df_preds[df_preds['train_valid'] == 'valid'].copy().reset_index(drop=True)

    # Créer les datasets AIF360
    dataset_true = create_aif360_dataset(df_val, 'label_binary', prot_attr)
    dataset_pred = dataset_true.copy(deepcopy=True)
    dataset_pred.labels = df_val['preds_binary'].values.reshape(-1, 1).astype(float)
    dataset_pred.scores = df_val['score_malade'].values.reshape(-1, 1)

    privileged_groups = [{prot_attr: 1}]
    unprivileged_groups = [{prot_attr: 0}]

    if method == 'ROC':
        postproc = RejectOptionClassification(
            unprivileged_groups=unprivileged_groups,
            privileged_groups=privileged_groups,
            low_class_thresh=0.01,
            high_class_thresh=0.99,
            num_class_thresh=100,
            num_ROC_margin=50,
            metric_name="Statistical parity difference",
            metric_ub=0.05,
            metric_lb=-0.05,
        )
        postproc = postproc.fit(dataset_true, dataset_pred)
        dataset_corrected = postproc.predict(dataset_pred)
        print(f"  ROC — Seuil optimal : {postproc.classification_threshold:.4f}")
        print(f"  ROC — Marge optimale : {postproc.ROC_margin:.4f}")

    elif method == 'CalibratedEqOdds':
        postproc = CalibratedEqOddsPostprocessing(
            privileged_groups=privileged_groups,
            unprivileged_groups=unprivileged_groups,
            cost_constraint="fnr",
            seed=42
        )
        postproc = postproc.fit(dataset_true, dataset_pred)
        dataset_corrected = postproc.predict(dataset_pred)

    # Calculer les métriques
    corrected_preds = dataset_corrected.labels[:, 0]

    metrics = get_metrics(
        y_true=df_val['label_binary'].values,
        y_pred=corrected_preds,
        prot_attr=df_val[prot_attr].values,
        priv_group=1,
        pos_label=1
    )
    metrics['method'] = label
    metrics['protected_attribute'] = prot_attr

    return metrics

print("✓ Fonction apply_post_processing définie.")

✓ Fonction apply_post_processing définie.


### 6.1 Reject Option Classification (ROC) — sur Baseline

In [60]:
# === ROC sur Baseline (Genre puis Position) ===
print("Application de ROC sur le modèle Baseline — attribut Genre...")
metrics_roc_baseline_gender = apply_post_processing(
    df_baseline, method='ROC', prot_attr='Gender_Binary', label='Baseline + ROC'
)
print_metrics(metrics_roc_baseline_gender, "Baseline + ROC — Fairness par Genre")

print("\nApplication de ROC sur le modèle Baseline — attribut Position...")
metrics_roc_baseline_pos = apply_post_processing(
    df_baseline, method='ROC', prot_attr='Position_Binary', label='Baseline + ROC'
)
print_metrics(metrics_roc_baseline_pos, "Baseline + ROC — Fairness par Position")

Application de ROC sur le modèle Baseline — attribut Genre...
  ROC — Seuil optimal : 0.4456
  ROC — Marge optimale : 0.0000

Baseline + ROC — Fairness par Genre
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.0337
  disparate_impact_ratio                        : 0.9253
  base_rate_preds                               : +0.4358
  equal_opportunity_difference                  : -0.0489
  average_odds_difference                       : -0.0518
  conditional_demographic_disparity             : -0.0025
  smoothed_edf                                  : +0.0775
  df_bias_amplification                         : -0.0591
  balanced_accuracy_score                       : +0.6539

Application de ROC sur le modèle Baseline — attribut Position...
  ROC — Seuil optimal : 0.7029
  ROC — Marge optimale : 0.0000

Baseline + ROC — Fairness par Position
  base_rate_truth                               : +0.4326
  statistical_parity_difference 

### 6.2 Calibrated Equalized Odds — sur Baseline

In [61]:
# === Calibrated EqOdds sur Baseline (Genre puis Position) ===
print("Application de Calibrated Equalized Odds sur le modèle Baseline — attribut Genre...")
metrics_ceqo_baseline_gender = apply_post_processing(
    df_baseline, method='CalibratedEqOdds', prot_attr='Gender_Binary', label='Baseline + CalEqOdds'
)
print_metrics(metrics_ceqo_baseline_gender, "Baseline + Calibrated EqOdds — Fairness par Genre")

print("\nApplication de Calibrated Equalized Odds sur le modèle Baseline — attribut Position...")
metrics_ceqo_baseline_pos = apply_post_processing(
    df_baseline, method='CalibratedEqOdds', prot_attr='Position_Binary', label='Baseline + CalEqOdds'
)
print_metrics(metrics_ceqo_baseline_pos, "Baseline + Calibrated EqOdds — Fairness par Position")

Application de Calibrated Equalized Odds sur le modèle Baseline — attribut Genre...

Baseline + Calibrated EqOdds — Fairness par Genre
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0552
  disparate_impact_ratio                        : 1.2223
  base_rate_preds                               : +0.2737
  equal_opportunity_difference                  : +0.0822
  average_odds_difference                       : +0.0478
  conditional_demographic_disparity             : +0.0051
  smoothed_edf                                  : +0.2003
  df_bias_amplification                         : +0.0638
  balanced_accuracy_score                       : +0.6005

Application de Calibrated Equalized Odds sur le modèle Baseline — attribut Position...

Baseline + Calibrated EqOdds — Fairness par Position
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.2073
  disparate_impact_ratio    

### 6.3 Combinaisons Pre-processing + Post-processing

Nous évaluons les combinaisons pre+post sur les deux axes sensibles:
- RW Genre + (ROC, CalEqOdds) évalué sur Genre et Position
- RW Position + (ROC, CalEqOdds) évalué sur Genre et Position

In [62]:
# === Combinaisons pre+post sur RW Genre et RW Position ===
required = ['df_gender', 'df_pos']
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(
        f"Variables manquantes pour les combinaisons pre+post: {missing}. "
        "Exécuter d'abord les cellules 18-25."
    )

print("Application de ROC sur RW Genre — attribut Genre...")
metrics_roc_rw_gender_gender = apply_post_processing(
    df_gender, method='ROC', prot_attr='Gender_Binary', label='RW Genre + ROC'
)
print_metrics(metrics_roc_rw_gender_gender, "RW Genre + ROC — Fairness par Genre")

print("\nApplication de ROC sur RW Genre — attribut Position...")
metrics_roc_rw_gender_pos = apply_post_processing(
    df_gender, method='ROC', prot_attr='Position_Binary', label='RW Genre + ROC'
)
print_metrics(metrics_roc_rw_gender_pos, "RW Genre + ROC — Fairness par Position")

print("\nApplication de CalEqOdds sur RW Genre — attribut Genre...")
metrics_ceqo_rw_gender_gender = apply_post_processing(
    df_gender, method='CalibratedEqOdds', prot_attr='Gender_Binary', label='RW Genre + CalEqOdds'
)
print_metrics(metrics_ceqo_rw_gender_gender, "RW Genre + CalEqOdds — Fairness par Genre")

print("\nApplication de CalEqOdds sur RW Genre — attribut Position...")
metrics_ceqo_rw_gender_pos = apply_post_processing(
    df_gender, method='CalibratedEqOdds', prot_attr='Position_Binary', label='RW Genre + CalEqOdds'
)
print_metrics(metrics_ceqo_rw_gender_pos, "RW Genre + CalEqOdds — Fairness par Position")

print("\nApplication de ROC sur RW Position — attribut Genre...")
metrics_roc_rw_pos_gender = apply_post_processing(
    df_pos, method='ROC', prot_attr='Gender_Binary', label='RW Position + ROC'
)
print_metrics(metrics_roc_rw_pos_gender, "RW Position + ROC — Fairness par Genre")

print("\nApplication de ROC sur RW Position — attribut Position...")
metrics_roc_rw_pos_pos = apply_post_processing(
    df_pos, method='ROC', prot_attr='Position_Binary', label='RW Position + ROC'
)
print_metrics(metrics_roc_rw_pos_pos, "RW Position + ROC — Fairness par Position")

print("\nApplication de CalEqOdds sur RW Position — attribut Genre...")
metrics_ceqo_rw_pos_gender = apply_post_processing(
    df_pos, method='CalibratedEqOdds', prot_attr='Gender_Binary', label='RW Position + CalEqOdds'
)
print_metrics(metrics_ceqo_rw_pos_gender, "RW Position + CalEqOdds — Fairness par Genre")

print("\nApplication de CalEqOdds sur RW Position — attribut Position...")
metrics_ceqo_rw_pos_pos = apply_post_processing(
    df_pos, method='CalibratedEqOdds', prot_attr='Position_Binary', label='RW Position + CalEqOdds'
)
print_metrics(metrics_ceqo_rw_pos_pos, "RW Position + CalEqOdds — Fairness par Position")

Application de ROC sur RW Genre — attribut Genre...
  ROC — Seuil optimal : 0.6336
  ROC — Marge optimale : 0.0000

RW Genre + ROC — Fairness par Genre
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0475
  disparate_impact_ratio                        : 1.1486
  base_rate_preds                               : +0.3419
  equal_opportunity_difference                  : +0.0428
  average_odds_difference                       : +0.0313
  conditional_demographic_disparity             : +0.0039
  smoothed_edf                                  : +0.1383
  df_bias_amplification                         : +0.0017
  balanced_accuracy_score                       : +0.6499

Application de ROC sur RW Genre — attribut Position...
  ROC — Seuil optimal : 0.6732
  ROC — Marge optimale : 0.0000

RW Genre + ROC — Fairness par Position
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0

### 6.4 Comparaison globale de toutes les approches

Nous rassemblons **toutes les configurations** testées dans un tableau et des graphiques comparatifs pour analyser le compromis performance / équité.

In [63]:
# === Tableau récapitulatif global ===
required = [
    'metrics_baseline_gender', 'metrics_baseline_pos',
    'metrics_rw_gender', 'metrics_rw_pos', 'metrics_rw_pos_gender',
    'metrics_rw_cross_gender', 'metrics_rw_cross_pos',
    'metrics_roc_baseline_gender', 'metrics_roc_baseline_pos',
    'metrics_ceqo_baseline_gender', 'metrics_ceqo_baseline_pos',
    'metrics_roc_rw_gender_gender', 'metrics_roc_rw_gender_pos',
    'metrics_ceqo_rw_gender_gender', 'metrics_ceqo_rw_gender_pos',
    'metrics_roc_rw_pos_gender', 'metrics_roc_rw_pos_pos',
    'metrics_ceqo_rw_pos_gender', 'metrics_ceqo_rw_pos_pos',
]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(
        f"Variables manquantes pour le tableau global: {missing}. "
        "Exécuter d'abord toutes les cellules d'entraînement, de post-processing et de combinaisons."
    )

all_metrics = [
    {**metrics_baseline_gender, 'method': 'Baseline', 'attr': 'Genre'},
    {**metrics_baseline_pos, 'method': 'Baseline', 'attr': 'Position'},
    {**metrics_rw_gender, 'method': 'RW Genre', 'attr': 'Genre'},
    {**metrics_rw_pos_gender, 'method': 'RW Position', 'attr': 'Genre'},
    {**metrics_rw_pos, 'method': 'RW Position', 'attr': 'Position'},
    {**metrics_rw_cross_gender, 'method': 'RW Genre×Position', 'attr': 'Genre'},
    {**metrics_rw_cross_pos, 'method': 'RW Genre×Position', 'attr': 'Position'},
    {**metrics_roc_baseline_gender, 'method': 'Baseline + ROC', 'attr': 'Genre'},
    {**metrics_roc_baseline_pos, 'method': 'Baseline + ROC', 'attr': 'Position'},
    {**metrics_ceqo_baseline_gender, 'method': 'Baseline + CalEqOdds', 'attr': 'Genre'},
    {**metrics_ceqo_baseline_pos, 'method': 'Baseline + CalEqOdds', 'attr': 'Position'},
    {**metrics_roc_rw_gender_gender, 'method': 'RW Genre + ROC', 'attr': 'Genre'},
    {**metrics_roc_rw_gender_pos, 'method': 'RW Genre + ROC', 'attr': 'Position'},
    {**metrics_ceqo_rw_gender_gender, 'method': 'RW Genre + CalEqOdds', 'attr': 'Genre'},
    {**metrics_ceqo_rw_gender_pos, 'method': 'RW Genre + CalEqOdds', 'attr': 'Position'},
    {**metrics_roc_rw_pos_gender, 'method': 'RW Position + ROC', 'attr': 'Genre'},
    {**metrics_roc_rw_pos_pos, 'method': 'RW Position + ROC', 'attr': 'Position'},
    {**metrics_ceqo_rw_pos_gender, 'method': 'RW Position + CalEqOdds', 'attr': 'Genre'},
    {**metrics_ceqo_rw_pos_pos, 'method': 'RW Position + CalEqOdds', 'attr': 'Position'},
]

df_compare = pd.DataFrame(all_metrics)
cols_display = [
    'method', 'attr', 'balanced_accuracy_score',
    'statistical_parity_difference', 'disparate_impact_ratio',
    'equal_opportunity_difference', 'average_odds_difference',
    'df_bias_amplification'
]

print("=" * 100)
print("TABLEAU RÉCAPITULATIF — Toutes les méthodes et les deux attributs sensibles")
print("=" * 100)
display(df_compare[cols_display].sort_values(['attr', 'method']).round(4))

TABLEAU RÉCAPITULATIF — Toutes les méthodes et les deux attributs sensibles


,method,attr,balanced_accuracy_score,statistical_parity_difference,disparate_impact_ratio,equal_opportunity_difference,average_odds_difference,df_bias_amplification
0,Baseline,Genre,0.6133,-0.0629,0.8282,-0.0727,-0.0762,0.0514
9,Baseline + CalEqOdds,Genre,0.6005,0.0552,1.2223,0.0822,0.0478,0.0638
7,Baseline + ROC,Genre,0.6539,-0.0337,0.9253,-0.0489,-0.0518,-0.0591
2,RW Genre,Genre,0.6353,0.0552,1.0926,0.0309,0.0384,0.0102
13,RW Genre + CalEqOdds,Genre,0.6351,0.0396,1.0665,0.0160,0.0228,-0.0335
11,RW Genre + ROC,Genre,0.6499,0.0475,1.1486,0.0428,0.0313,0.0017
5,RW Genre×Position,Genre,0.6282,0.0103,1.0242,-0.0196,-0.0066,-0.1126
3,RW Position,Genre,0.6267,0.0790,1.3913,0.0414,0.0616,0.1928
17,RW Position + CalEqOdds,Genre,0.6267,0.0790,1.3913,0.0414,0.0616,0.1928
15,RW Position + ROC,Genre,0.6125,0.0490,1.0670,-0.0459,0.0265,0.0633


In [64]:
# === Visualisation comparative multi-attributs ===
if 'df_compare' not in globals():
    raise RuntimeError("df_compare absent. Exécuter d'abord la cellule 40 (tableau global).")

df_plot = df_compare.copy()

fig1 = px.bar(
    df_plot,
    x='method', y='balanced_accuracy_score',
    color='attr', barmode='group',
    title="Balanced Accuracy par méthode et attribut sensible",
    labels={'balanced_accuracy_score': 'Balanced Accuracy', 'method': 'Méthode', 'attr': 'Attribut'}
)
fig1.update_layout(height=500, template='plotly_white')
fig1.show()

fig2 = px.bar(
    df_plot,
    x='method', y='statistical_parity_difference',
    color='attr', barmode='group',
    title="Statistical Parity Difference par méthode (proche de 0 = mieux)",
    labels={'statistical_parity_difference': 'SPD', 'method': 'Méthode', 'attr': 'Attribut'}
)
fig2.add_hline(y=0, line_dash='dash', line_color='black')
fig2.add_hline(y=0.05, line_dash='dot', line_color='red')
fig2.add_hline(y=-0.05, line_dash='dot', line_color='red')
fig2.update_layout(height=500, template='plotly_white')
fig2.show()

fig3 = px.bar(
    df_plot,
    x='method', y='disparate_impact_ratio',
    color='attr', barmode='group',
    title="Disparate Impact Ratio par méthode (proche de 1 = mieux)",
    labels={'disparate_impact_ratio': 'DIR', 'method': 'Méthode', 'attr': 'Attribut'}
)
fig3.add_hline(y=1.0, line_dash='dash', line_color='black')
fig3.add_hline(y=1.25, line_dash='dot', line_color='red')
fig3.add_hline(y=0.8, line_dash='dot', line_color='red')
fig3.update_layout(height=500, template='plotly_white')
fig3.show()

In [65]:
# === Synthèse triée pour lecture rapide ===
if 'df_compare' not in globals():
    raise RuntimeError("df_compare absent. Exécuter d'abord la cellule 40 (tableau global).")

for attr in ['Genre', 'Position']:
    print(f"\n=== Top méthodes pour {attr} ===")
    df_attr = df_compare[df_compare['attr'] == attr].copy()
    df_attr['score_tradeoff'] = (
        df_attr['balanced_accuracy_score']
        - 0.5 * df_attr['statistical_parity_difference'].abs()
        - 0.5 * (df_attr['disparate_impact_ratio'] - 1.0).abs()
    )
    keep = ['method', 'balanced_accuracy_score', 'statistical_parity_difference', 'disparate_impact_ratio', 'score_tradeoff']
    display(df_attr[keep].sort_values('score_tradeoff', ascending=False).head(5).round(4))


=== Top méthodes pour Genre ===


,method,balanced_accuracy_score,statistical_parity_difference,disparate_impact_ratio,score_tradeoff
5,RW Genre×Position,0.6282,0.0103,1.0242,0.6110
7,Baseline + ROC,0.6539,-0.0337,0.9253,0.5996
13,RW Genre + CalEqOdds,0.6351,0.0396,1.0665,0.5821
2,RW Genre,0.6353,0.0552,1.0926,0.5614
15,RW Position + ROC,0.6125,0.0490,1.0670,0.5545



=== Top méthodes pour Position ===


,method,balanced_accuracy_score,statistical_parity_difference,disparate_impact_ratio,score_tradeoff
14,RW Genre + CalEqOdds,0.6261,0.0115,1.0203,0.6102
16,RW Position + ROC,0.6810,0.0401,1.1040,0.6089
12,RW Genre + ROC,0.6262,0.0359,1.1507,0.5328
6,RW Genre×Position,0.6282,-0.1456,0.6986,0.4047
4,RW Position,0.6267,-0.1021,0.6317,0.3915


---
## 7. Analyse et Discussion

### Méthode d'interprétation

L'analyse doit être lue à partir des sorties réellement exécutées dans les cellules de comparaison globale. Les conclusions ci-dessous sont formulées de manière conditionnelle et doivent être adaptées aux chiffres effectivement observés.

### Lecture du compromis performance / équité

1. Performance: la balanced accuracy mesure l'utilité clinique brute.
2. Équité: SPD proche de 0 et DIR proche de 1 indiquent une meilleure parité entre groupes.
3. Trade-off: une méthode peut améliorer l'équité mais dégrader la performance; l'objectif est une zone de compromis acceptable.

### Ce qu'on évalue explicitement dans ce notebook

- Pre-processing:
  - Baseline
  - RW Genre
  - RW Position
  - RW Genre × Position
- Post-processing:
  - ROC
  - Calibrated EqOdds
- Combinaisons pre+post:
  - RW Genre + (ROC, CalEqOdds)
  - RW Position + (ROC, CalEqOdds)
- Deux attributs sensibles:
  - Genre
  - Position

### Interprétation clinique prudente

En contexte médical, les écarts de faux négatifs entre groupes sont critiques. Une méthode peut être préférée même si la performance baisse légèrement, si elle réduit un biais cliniquement important. Le choix final doit être fait à partir du tableau global et des graphes, et non d'une affirmation a priori.

---
## 8. Conclusion

Ce notebook propose un protocole complet conforme au cadrage du projet final:
- analyse des déséquilibres,
- plusieurs pre-processing,
- plusieurs post-processing,
- combinaisons pre+post,
- comparaison multi-critères sur deux attributs sensibles.

### Conclusion à renseigner après exécution complète

La conclusion finale doit s'appuyer sur les sorties de la section 6.4:
1. Méthode la plus équilibrée pour le Genre (au vu de BA, SPD, DIR, EOD, AOD).
2. Méthode la plus équilibrée pour la Position.
3. Discussion du compromis retenu et justification clinique.

### Point de rigueur

Si toutes les cellules d'entraînement/comparaison n'ont pas été exécutées dans l'état sauvegardé, il faut éviter toute affirmation forte sur "la meilleure méthode". Le notebook est maintenant structuré pour produire ces preuves de façon reproductible.